# Notebook 1 : Prétraitement des Images
## Projet Le Refuge - Classification de races de chiens

Dans ce notebook, on va :
1. Télécharger le dataset Stanford Dogs
2. Sélectionner 3 races de chiens
3. Préparer les images (resize, normalisation)
4. Mettre en place la data augmentation

## 1. Import des bibliothèques

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import pickle
import os
from os import listdir
import urllib.request
import tarfile
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from glob import glob
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Rescaling, RandomFlip, RandomRotation, RandomZoom
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


## 2. Téléchargement du dataset Stanford Dogs

In [ ]:

# Chargement des 120 races
path_base = "Images"

def load_all_dogs(base_path):
    data_list = []
    all_folders = [f for f in os.listdir(base_path) 
                   if os.path.isdir(os.path.join(base_path, f))]
    
    print(f"Nombre de races : {len(all_folders)}\n")
    
    for folder in all_folders:
        full_path = os.path.join(base_path, folder)
        images = glob(os.path.join(full_path, "*.jpg"))
        breed_name = folder.split('-')[-1]
        
        for img in images:
            data_list.append({"image_path": img, "label_name": breed_name})
    
    return pd.DataFrame(data_list)

data_full = load_all_dogs(path_base)

# Encoding des labels
le = LabelEncoder()
data_full['label'] = le.fit_transform(data_full['label_name'])

num_classes = data_full['label'].nunique()
print(f"\nNombre de classes : {num_classes}")
print(f"Total images : {len(data_full)}")

# Split en 3 : Train (70%) / Val (15%) / Test (15%)
# Étape 1 : Train vs (Val+Test)
train_df, temp_df = train_test_split(
    data_full, 
    test_size=0.3,  # 30% pour val+test
    stratify=data_full['label'], 
    random_state=42
)

# Étape 2 : Val vs Test (on split les 30% en deux parts égales)
val_df, test_df = train_test_split(
    temp_df, 
    test_size=0.5,  # 50% de 30% = 15% du total
    stratify=temp_df['label'], 
    random_state=42
)

Nombre de races : 120


Nombre de classes : 119
Total images : 20580

=== Répartition des données ===
Train : 14406 images (70.0%)
Val   : 3087 images (15.0%)
Test  : 3087 images (15.0%)
Total : 20580 images


# Générateurs de données pour les 3 jeux train/val/test

In [ ]:
# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32

# Data augmentation UNIQUEMENT pour le train
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.15
)

# On rescale juste pour val et test
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Générateur TRAIN
train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col='image_path',
    y_col='label_name',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

# Générateur VALIDATION
val_generator = val_test_datagen.flow_from_dataframe(
    val_df,
    x_col='image_path',
    y_col='label_name',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # Pas de shuffle pour val
)

# Générateur TEST
test_generator = val_test_datagen.flow_from_dataframe(
    test_df,
    x_col='image_path',
    y_col='label_name',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False  # Pas de shuffle pour test
)

print(f"\nNombre de batches par epoch :")
print(f"Train : {len(train_generator)}")
print(f"Val   : {len(val_generator)}")
print(f"Test  : {len(test_generator)}")

Found 14406 validated image filenames belonging to 119 classes.
Found 3087 validated image filenames belonging to 119 classes.
Found 3087 validated image filenames belonging to 119 classes.

Nombre de batches par epoch :
Train : 451
Val   : 97
Test  : 97


# Construction du modèle choisi dans la projet précédent : MobileNetV2

In [11]:

# Base MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Gel des couches de base
base_model.trainable = False

# Couches de classification
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model_mobilenet = Model(inputs=base_model.input, outputs=predictions)

# Compilation
model_mobilenet.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_accuracy')
    ]
)

print(f"\n=== Architecture MobileNetV2 ===")
print(f"Paramètres totaux : {model_mobilenet.count_params():,}")
print(f"Paramètres entraînables : {sum([tf.size(w).numpy() for w in model_mobilenet.trainable_weights]):,}")


=== Architecture MobileNetV2 ===
Paramètres totaux : 2,974,903
Paramètres entraînables : 716,919


# Entraînement avec validation

In [12]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    # Arrêt si val_loss ne s'améliore plus
    EarlyStopping(
        monitor='val_loss', 
        patience=10, 
        restore_best_weights=True,
        verbose=1
    ),
    
    # Réduction du learning rate
    ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=5, 
        min_lr=1e-7,
        verbose=1
    ),
    
    # Sauvegarde du meilleur modèle
    ModelCheckpoint(
        'best_mobilenet_120races.h5', 
        save_best_only=True, 
        monitor='val_accuracy',
        verbose=1
    )
]

# Entraînement
history_mobilenet = model_mobilenet.fit(
    train_generator,
    validation_data=val_generator,  # Validation sur val_generator
    epochs=50,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 693ms/step - accuracy: 0.2888 - loss: 3.1258 - top5_accuracy: 0.5481
Epoch 1: val_accuracy improved from None to 0.69971, saving model to best_mobilenet_120races.h5



Epoch 1: finished saving model to best_mobilenet_120races.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 360s 788ms/step - accuracy: 0.4196 - loss: 2.3076 - top5_accuracy: 0.7234 - val_accuracy: 0.6997 - val_loss: 1.0226 - val_top5_accuracy: 0.9443 - learning_rate: 0.0010
Epoch 2/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 693ms/step - accuracy: 0.5764 - loss: 1.4902 - top5_accuracy: 0.8771
Epoch 2: val_accuracy improved from 0.69971 to 0.70327, saving model to best_mobilenet_120races.h5



Epoch 2: finished saving model to best_mobilenet_120races.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 352s 781ms/step - accuracy: 0.5740 - loss: 1.4962 - top5_accuracy: 0.8749 - val_accuracy: 0.7033 - val_loss: 0.9572 - val_top5_accuracy: 0.9504 - learning_rate: 0.0010
Epoch 3/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 676ms/step - accuracy: 0.6086 - loss: 1.3419 - top5_accuracy: 0.8948
Epoch 3: val_accuracy improved from 0.70327 to 0.72627, saving model to best_mobilenet_120races.h5



Epoch 3: finished saving model to best_mobilenet_120races.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 344s 763ms/step - accuracy: 0.6077 - loss: 1.3592 - top5_accuracy: 0.8933 - val_accuracy: 0.7263 - val_loss: 0.9168 - val_top5_accuracy: 0.9459 - learning_rate: 0.0010
Epoch 4/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 682ms/step - accuracy: 0.6394 - loss: 1.2447 - top5_accuracy: 0.9102
Epoch 4: val_accuracy did not improve from 0.72627
451/451 ━━━━━━━━━━━━━━━━━━━━ 346s 767ms/step - accuracy: 0.6313 - loss: 1.2716 - top5_accuracy: 0.9032 - val_accuracy: 0.7224 - val_loss: 0.9193 - val_top5_accuracy: 0.9469 - learning_rate: 0.0010
Epoch 5/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 687ms/step - accuracy: 0.6450 - loss: 1.2186 - top5_accuracy: 0.9144
Epoch 5: val_accuracy improved from 0.72627 to 0.72692, saving model to best_mobilenet_120races.h5



Epoch 5: finished saving model to best_mobilenet_120races.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 349s 775ms/step - accuracy: 0.6419 - loss: 1.2326 - top5_accuracy: 0.9109 - val_accuracy: 0.7269 - val_loss: 0.9033 - val_top5_accuracy: 0.9449 - learning_rate: 0.0010
Epoch 6/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 694ms/step - accuracy: 0.6426 - loss: 1.1972 - top5_accuracy: 0.9142
Epoch 6: val_accuracy improved from 0.72692 to 0.72724, saving model to best_mobilenet_120races.h5


...


Epoch 7: finished saving model to best_mobilenet_120races.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 345s 764ms/step - accuracy: 0.6556 - loss: 1.1634 - top5_accuracy: 0.9204 - val_accuracy: 0.7334 - val_loss: 0.9093 - val_top5_accuracy: 0.9446 - learning_rate: 0.0010
Epoch 8/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 687ms/step - accuracy: 0.6721 - loss: 1.1024 - top5_accuracy: 0.9237
Epoch 8: val_accuracy did not improve from 0.73340
451/451 ━━━━━━━━━━━━━━━━━━━━ 349s 774ms/step - accuracy: 0.6657 - loss: 1.1247 - top5_accuracy: 0.9217 - val_accuracy: 0.7292 - val_loss: 0.9173 - val_top5_accuracy: 0.9449 - learning_rate: 0.0010
Epoch 9/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 678ms/step - accuracy: 0.6842 - loss: 1.0679 - top5_accuracy: 0.9280
Epoch 9: val_accuracy did not improve from 0.73340
451/451 ━━━━━━━━━━━━━━━━━━━━ 347s 768ms/step - accuracy: 0.6776 - loss: 1.0890 - top5_accuracy: 0.9272 - val_accuracy: 0.7292 - val_loss: 0.9181 - val_top5_accuracy: 0.9527 - learning_rate: 0.0010
Epoch 10/50
451/451 ━


Epoch 11: finished saving model to best_mobilenet_120races.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 346s 766ms/step - accuracy: 0.7164 - loss: 0.9462 - top5_accuracy: 0.9421 - val_accuracy: 0.7431 - val_loss: 0.8718 - val_top5_accuracy: 0.9498 - learning_rate: 5.0000e-04
Epoch 12/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 676ms/step - accuracy: 0.7244 - loss: 0.9079 - top5_accuracy: 0.9485
Epoch 12: val_accuracy did not improve from 0.74312
451/451 ━━━━━━━━━━━━━━━━━━━━ 344s 762ms/step - accuracy: 0.7260 - loss: 0.9095 - top5_accuracy: 0.9475 - val_accuracy: 0.7402 - val_loss: 0.8854 - val_top5_accuracy: 0.9504 - learning_rate: 5.0000e-04
Epoch 13/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 671ms/step - accuracy: 0.7355 - loss: 0.8566 - top5_accuracy: 0.9526
Epoch 13: val_accuracy did not improve from 0.74312
451/451 ━━━━━━━━━━━━━━━━━━━━ 342s 757ms/step - accuracy: 0.7383 - loss: 0.8588 - top5_accuracy: 0.9518 - val_accuracy: 0.7425 - val_loss: 0.8775 - val_top5_accuracy: 0.9534 - learning_rate: 5.0000e-04
Epoc


Epoch 15: finished saving model to best_mobilenet_120races.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 402s 891ms/step - accuracy: 0.7412 - loss: 0.8396 - top5_accuracy: 0.9539 - val_accuracy: 0.7493 - val_loss: 0.8685 - val_top5_accuracy: 0.9537 - learning_rate: 5.0000e-04
Epoch 16/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7538 - loss: 0.8027 - top5_accuracy: 0.9535
Epoch 16: val_accuracy did not improve from 0.74927
451/451 ━━━━━━━━━━━━━━━━━━━━ 590s 1s/step - accuracy: 0.7508 - loss: 0.8071 - top5_accuracy: 0.9570 - val_accuracy: 0.7467 - val_loss: 0.8672 - val_top5_accuracy: 0.9511 - learning_rate: 5.0000e-04
Epoch 17/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7618 - loss: 0.7604 - top5_accuracy: 0.9585
Epoch 17: val_accuracy did not improve from 0.74927
451/451 ━━━━━━━━━━━━━━━━━━━━ 517s 1s/step - accuracy: 0.7549 - loss: 0.7836 - top5_accuracy: 0.9572 - val_accuracy: 0.7454 - val_loss: 0.8855 - val_top5_accuracy: 0.9501 - learning_rate: 5.0000e-04
Epoch 18/50
451/


Epoch 23: finished saving model to best_mobilenet_120races.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 396s 878ms/step - accuracy: 0.7931 - loss: 0.6580 - top5_accuracy: 0.9707 - val_accuracy: 0.7519 - val_loss: 0.8719 - val_top5_accuracy: 0.9559 - learning_rate: 2.5000e-04
Epoch 24/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 763ms/step - accuracy: 0.8053 - loss: 0.6176 - top5_accuracy: 0.9752
Epoch 24: val_accuracy improved from 0.75186 to 0.75640, saving model to best_mobilenet_120races.h5



Epoch 24: finished saving model to best_mobilenet_120races.h5
451/451 ━━━━━━━━━━━━━━━━━━━━ 394s 874ms/step - accuracy: 0.7986 - loss: 0.6315 - top5_accuracy: 0.9744 - val_accuracy: 0.7564 - val_loss: 0.8754 - val_top5_accuracy: 0.9530 - learning_rate: 2.5000e-04
Epoch 25/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 795ms/step - accuracy: 0.7985 - loss: 0.6246 - top5_accuracy: 0.9739
Epoch 25: val_accuracy did not improve from 0.75640
451/451 ━━━━━━━━━━━━━━━━━━━━ 399s 885ms/step - accuracy: 0.7966 - loss: 0.6345 - top5_accuracy: 0.9738 - val_accuracy: 0.7545 - val_loss: 0.8688 - val_top5_accuracy: 0.9504 - learning_rate: 2.5000e-04
Epoch 26/50
451/451 ━━━━━━━━━━━━━━━━━━━━ 0s 872ms/step - accuracy: 0.8048 - loss: 0.6166 - top5_accuracy: 0.9750
Epoch 26: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 26: val_accuracy did not improve from 0.75640
451/451 ━━━━━━━━━━━━━━━━━━━━ 441s 978ms/step - accuracy: 0.7969 - loss: 0.6330 - top5_accuracy: 0.9739 - val_accuracy: 0.7545 -

# Evaluation pour les 3 jeux

In [13]:
# Évaluation sur TRAIN
train_loss, train_acc, train_top5 = model_mobilenet.evaluate(train_generator, verbose=0)

# Évaluation sur VAL
val_loss, val_acc, val_top5 = model_mobilenet.evaluate(val_generator, verbose=0)

# Évaluation sur TEST (données jamais vues)
test_loss, test_acc, test_top5 = model_mobilenet.evaluate(test_generator, verbose=0)

print(f"\n{'='*60}")
print(f"{'=== MobileNetV2 - Résultats finaux ===':^60}")
print(f"{'='*60}")
print(f"{'Dataset':<10} {'Loss':<12} {'Accuracy':<12} {'Top-5 Acc':<12}")
print(f"{'-'*60}")
print(f"{'Train':<10} {train_loss:<12.4f} {train_acc:<12.4f} {train_top5:<12.4f}")
print(f"{'Val':<10} {val_loss:<12.4f} {val_acc:<12.4f} {val_top5:<12.4f}")
print(f"{'Test':<10} {test_loss:<12.4f} {test_acc:<12.4f} {test_top5:<12.4f}")
print(f"{'='*60}")

# Vérification d'overfitting
if train_acc - val_acc > 0.1:
    print("  Attention : Possible overfitting détecté (écart train-val > 10%)")
if val_acc - test_acc > 0.05:
    print("  Attention : Écart val-test significatif")


           === MobileNetV2 - Résultats finaux ===           
Dataset    Loss         Accuracy     Top-5 Acc   
------------------------------------------------------------
Train      0.5130       0.8372       0.9833      
Val        0.8672       0.7467       0.9511      
Test       0.7793       0.7600       0.9563      


In [14]:

with open('label_encoder_mobilenet.pkl', 'wb') as f:
    pickle.dump(le, f)

print(" LabelEncoder sauvegardé : label_encoder_mobilenet.pkl")

 LabelEncoder sauvegardé : label_encoder_mobilenet.pkl
